In [1]:
import os, sys, json, re, time, subprocess
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
except Exception:
    import sys as _s, subprocess as _sp
    _sp.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'mlflow', 'fastapi', 'uvicorn', 'joblib', 'requests'])
    import pandas as pd
    import numpy as np

p = Path.cwd()
while not (p / 'churnDataset.csv').exists() and p.parent != p:
    p = p.parent
os.chdir(p)
rt = Path.cwd()
dd = rt / 'data'
md = rt / 'models'
ld = rt / 'logs'
mlr = rt / 'mlruns'
for d in [dd, md, ld]:
    d.mkdir(exist_ok=True)

import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(mlr.resolve().as_uri())
rs = json.loads((md / 'runs.json').read_text(encoding='utf-8'))
rs = sorted(rs, key=lambda r: r['val_f1'], reverse=True)
mn = 'churn_model'
cl = MlflowClient()
vs = []
for r in rs:
    mv = mlflow.register_model(f"runs:/{r['run_id']}/model", mn)
    vs.append({'name': r['name'], 'version': mv.version, 'val_f1': r['val_f1'], 'test_f1': r['test_f1']})
for i, v in enumerate(vs):
    st = 'Production' if i == 0 else 'Staging'
    try:
        cl.transition_model_version_stage(mn, v['version'], st, archive_existing_versions=False)
    except Exception:
        cl.set_registered_model_alias(mn, st.lower(), v['version'])
    v['stage'] = st
(md / 'registry.json').write_text(json.dumps(vs, indent=2), encoding='utf-8')
print('production', vs[0]['name'], vs[0]['version'], round(vs[0]['val_f1'], 4))
if len(vs) > 1:
    print('compare', vs[0]['name'], 'better than', vs[1]['name'], 'by val_f1')
vs


C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Successfully registered model 'churn_model'.
2026/05/04 13:29:11 WARNING mlflow.tracking._model_registry.fluent: Run with id fa0cc00f9aa44cf99da40e8c1ce663fb has no artifacts at artifact path 'model', registering model based on models:/m-7195f09b48d04ead97f423b74c2d8d9d instead


Created version '1' of model 'churn_model'.
Registered model 'churn_model' already exists. Creating a new version of this model...
2026/05/04 13:29:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 2ce8b2ad473e4e6990692658e1f241f8 has no artifacts at artifact path 'model', registering model based on models:/m-e57b4857ee82497c967b5159b7f24207 instead


Created version '2' of model 'churn_model'.


C:\Users\Admin\AppData\Local\Temp\ipykernel_17480\1354673397.py:40: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  cl.transition_model_version_stage(mn, v['version'], st, archive_existing_versions=False)


production rf 1 0.9974
compare rf better than lr by val_f1


[{'name': 'rf',
  'version': 1,
  'val_f1': 0.9974269713939761,
  'test_f1': 0.9971251323952186,
  'stage': 'Production'},
 {'name': 'lr',
  'version': 2,
  'val_f1': 0.8697092935683085,
  'test_f1': 0.8668478260869565,
  'stage': 'Staging'}]